# InsightEngine: Used Car Price Prediction

## Phase II: Data Preprocessing

### Objective

The objective of this phase is to clean and prepare the dataset for machine learning by performing feature engineering, encoding categorical variables, and splitting the dataset into training and testing sets.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/raw/car_data.csv")

df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [3]:
print(df.shape)

df.info()

(301, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Seller_Type    301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB


## Step 5: Create a New Feature – Car Age

The dataset contains the **Year** feature, which represents the manufacturing year of the vehicle.

Instead of using the manufacturing year directly, we create a new feature called **Car_Age**, which represents the age of the vehicle in years.

### Why create Car_Age?

- Vehicle age has a direct impact on its resale value.
- Older cars generally have lower selling prices due to depreciation.
- Machine learning models often learn better from the vehicle's age than from the manufacturing year.

In this project, the current year is assumed to be **2025** for calculating the car's age.

In [4]:
# Create a new feature: Car_Age

current_year = 2025

df["Car_Age"] = current_year - df["Year"]

# Display the first five records
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0,11
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0,12
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0,8
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0,14
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0,11


## Step 6: Drop Unnecessary Columns

After creating the **Car_Age** feature, the original **Year** column becomes redundant and is removed.

The **Car_Name** column is also dropped because it is a text identifier with many unique values. It does not directly contribute to predicting the selling price in this baseline regression model.

Removing unnecessary columns simplifies the dataset and helps reduce model complexity.

In [5]:
# Drop unnecessary columns

df.drop(["Car_Name", "Year"], axis=1, inplace=True)

# Display the updated dataset
df.head()

,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,3.35,5.59,27000,Petrol,Dealer,Manual,0,11
1,4.75,9.54,43000,Diesel,Dealer,Manual,0,12
2,7.25,9.85,6900,Petrol,Dealer,Manual,0,8
3,2.85,4.15,5200,Petrol,Dealer,Manual,0,14
4,4.60,6.87,42450,Diesel,Dealer,Manual,0,11


## Step 7: Encode Categorical Variables

Machine learning algorithms require numerical input. However, the dataset contains three categorical features:

- Fuel_Type
- Seller_Type
- Transmission

These categorical variables are converted into numerical form using **One-Hot Encoding**.

One-Hot Encoding creates separate binary (0 or 1) columns for each category. To avoid multicollinearity (the dummy variable trap), the first category is dropped using `drop_first=True`.

In [6]:
# Check the unique categories in each categorical feature

categorical_columns = ["Fuel_Type", "Seller_Type", "Transmission"]

for column in categorical_columns:
    print(f"{column}:")
    print(df[column].unique())
    print("-" * 40)

Fuel_Type:
['Petrol' 'Diesel' 'CNG']
----------------------------------------
Seller_Type:
['Dealer' 'Individual']
----------------------------------------
Transmission:
['Manual' 'Automatic']
----------------------------------------


In [7]:
# Convert categorical variables into numerical variables

df = pd.get_dummies(
    df,
    columns=["Fuel_Type", "Seller_Type", "Transmission"],
    drop_first=True,
    dtype=int
)

# Display the first five records
df.head()

,Selling_Price,Present_Price,Kms_Driven,Owner,Car_Age,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,3.35,5.59,27000,0,11,0,1,0,1
1,4.75,9.54,43000,0,12,1,0,0,1
2,7.25,9.85,6900,0,8,0,1,0,1
3,2.85,4.15,5200,0,14,0,1,0,1
4,4.60,6.87,42450,0,11,1,0,0,1


## Step 8: Define Features and Target Variable

In supervised machine learning, the dataset is divided into:

- **Features (X):** Independent variables used to predict the output.
- **Target (y):** Dependent variable that the model aims to predict.

For this project:

- **Target Variable:** `Selling_Price`
- **Features:** All remaining columns after preprocessing.

Separating the features and target variable is an essential step before training machine learning models.

In [8]:
# Define the feature matrix (X) and target variable (y)

X = df.drop("Selling_Price", axis=1)

y = df["Selling_Price"]

# Display the dimensions
print("Feature Matrix Shape (X):", X.shape)
print("Target Variable Shape (y):", y.shape)

Feature Matrix Shape (X): (301, 8)
Target Variable Shape (y): (301,)


In [9]:
# Display the first five rows of the feature matrix
X.head()

,Present_Price,Kms_Driven,Owner,Car_Age,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,5.59,27000,0,11,0,1,0,1
1,9.54,43000,0,12,1,0,0,1
2,9.85,6900,0,8,0,1,0,1
3,4.15,5200,0,14,0,1,0,1
4,6.87,42450,0,11,1,0,0,1


In [10]:
# Display the first five values of the target variable
y.head()

0    3.35
1    4.75
2    7.25
3    2.85
4    4.60
Name: Selling_Price, dtype: float64

Explanation
X contains all the input features used to predict the car's selling price.
y contains the actual selling prices that the model will learn to predict.
These variables will be used in the next step to split the dataset into training and testing sets.

## Step 9: Split the Dataset into Training and Testing Sets

Before training a machine learning model, the dataset is divided into two parts:

- **Training Set:** Used to train the machine learning model.
- **Testing Set:** Used to evaluate the performance of the trained model on unseen data.

In this project:

- **80%** of the data is used for training.
- **20%** of the data is used for testing.

The `random_state=42` parameter ensures that the data split is reproducible, meaning the same training and testing sets will be generated every time the code is executed.

In [11]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Feature Set :", X_train.shape)
print("Testing Feature Set  :", X_test.shape)
print("Training Target Set  :", y_train.shape)
print("Testing Target Set   :", y_test.shape)

Training Feature Set : (240, 8)
Testing Feature Set  : (61, 8)
Training Target Set  : (240,)
Testing Target Set   : (61,)


This observation summarizes everything you completed in Phase II:
Data loading
Feature engineering (Car_Age)
Dropping unnecessary columns
Encoding categorical variables
Defining features and target
Train-test split

# Phase II Observations

- The dataset was successfully loaded without any missing or inconsistent values.
- A new feature, **Car_Age**, was created from the **Year** column to better represent the age of the vehicle, which is more relevant for predicting the selling price.
- The original **Year** and **Car_Name** columns were removed as they were either redundant or not directly useful for regression modeling.
- Categorical features (**Fuel_Type**, **Seller_Type**, and **Transmission**) were converted into numerical format using **One-Hot Encoding**, making the dataset suitable for machine learning algorithms.
- The target variable (**Selling_Price**) was separated from the input features to prepare the dataset for supervised learning.
- The dataset was split into **80% training data** and **20% testing data** using a fixed random state to ensure reproducibility.
- The preprocessed dataset is now clean, numerical, and ready for feature engineering and regression model training.